In [ ]:
import cv2
import cvzone
import os
import numpy as np

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))

def nothing(x):
    pass

cv2.namedWindow("Track Bars")
cv2.resizeWindow("Track Bars", 650, 210)
cv2.moveWindow("Track Bars", 0, 0)

cv2.createTrackbar("Red", "Track Bars", 0, 255, nothing)
cv2.createTrackbar("Blue", "Track Bars", 0, 255, nothing)
cv2.createTrackbar("Green", "Track Bars", 0, 255, nothing)

img_path1 = os.path.join(script_dir, "Images", "Apricots.jpg")
img = cv2.imread(img_path1)

if img is not None:
    img = cv2.resize(img, (320, 240))
    imgHSV = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
else:
    print("Failed to Load Image, Check Path")
    imgHSV = None

cv2.namedWindow("Original")
cv2.namedWindow("HSV Format")
cv2.namedWindow("Mask Layer")
cv2.namedWindow("Result Output")

cv2.moveWindow("Original", 0, 280)
cv2.moveWindow("HSV Format", 330, 280)
cv2.moveWindow("Mask Layer", 660, 280)
cv2.moveWindow("Result Output", 990, 280)

while True:
    red_val = cv2.getTrackbarPos("Red", "Track Bars")
    blue_val = cv2.getTrackbarPos("Blue", "Track Bars")
    green_val = cv2.getTrackbarPos("Green", "Track Bars")

    lower = np.array([max(0, blue_val - 40), max(0, green_val - 40), max(0, red_val - 40)])
    upper = np.array([min(255, blue_val + 40), min(255, green_val + 40), min(255, red_val + 40)])
    
    mask = cv2.inRange(img, lowerb=lower, upperb=upper)
    imgResult = cv2.bitwise_and(img, img, mask=mask)
    
    try:
        if img is not None:
            cv2.imshow("Original", img)
            cv2.imshow("HSV Format", imgHSV)
            cv2.imshow("Mask Layer", mask)
            cv2.imshow("Result Output", imgResult)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            print(f"Failed to Display Images, They don't Exist")
            break
    except BaseException as e:
        print(f"Failed to Display Images, An Error has Occured = {e}")
        break

cv2.destroyAllWindows()

In [5]:
import cv2
import os

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
shapes_path = os.path.join(script_dir, "Images", "Shapes.jpg")
shapes = cv2.imread(shapes_path)

grayed = cv2.cvtColor(shapes, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(grayed, ksize=(3,3), sigmaX=0)
edged = cv2.Canny(blur, threshold1=50, threshold2=150)

imgContour = shapes.copy()

def getShapes(image):
    contours, hiearchy = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    for shape in contours:
        area = cv2.contourArea(shape)
        
        cv2.drawContours(imgContour, shape, -1, (23, 22, 22), thickness=3)

        if area > 450:
            perimeter = cv2.arcLength(shape, True)
            approx = cv2.approxPolyDP(shape, 0.02 * perimeter, True) 
            corners = len(approx)
            
            x, y, width, height = cv2.boundingRect(approx)

            if corners == 3:
                objecttype = "Triangle"
            elif corners == 4:
                aspect_ratio = width / float(height)
                if 0.95 <= aspect_ratio <= 1.05:
                    objecttype = "Square"
                else:
                    objecttype = "Rectangle"
            elif corners == 5:
                objecttype = "Pentagon"
            elif corners == 6:
                objecttype = "Hexagon"
            elif corners == 7:
                objecttype = "Heptagon"
            elif corners == 8:
                objecttype = "Octagon"
            elif corners > 8:
                objecttype = "Circle / Polygon"
            else:
                objecttype = "None"                                        

            cv2.rectangle(imgContour, (x, y), (x + width, y + height), color=(153, 191, 224), thickness=3)
            cv2.putText(imgContour, objecttype, 
                        (x + 10, y + 25), cv2.FONT_HERSHEY_COMPLEX, fontScale=0.4, color=(15,15,15), thickness=1)

getShapes(edged)

cv2.imshow("Original Shapes", shapes)
cv2.imshow("Grayed Shapes", grayed)
cv2.imshow("Blurred Shapes", blur)
cv2.imshow("Edged Shapes", edged)
cv2.imshow("Outlined Shapes", imgContour)
cv2.waitKey(0)
cv2.destroyAllWindows()